# **Part 1:** Import Libraries, Load & Preprocess Data

In [12]:
# Import necessary libaries
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.optimizers import Adam, SGD
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from tensorflow.keras.regularizers import l2 # Needed for L2 regularization task

import matplotlib.pyplot as plt
from google.colab import files

# Random seed for reproduceability
tf.random.set_seed(42)

# Load the datasets
files.upload()            # Note: When prompted to upload files, select both the sign_mnist_train.csv & the sign_mnist_test.csv files
print("Loading datasets...")
train = pd.read_csv("sign_mnist_train.csv")
test = pd.read_csv("sign_mnist_test.csv")

# Seperate the Features (X) and Labels (y)
X_train = train.drop("label", axis=1).values
y_train = train["label"].values

X_test = test.drop("label", axis=1).values
y_test = test["label"].values

# Normalize the pixel values (scale them from 0-255 to 0-1)
X_train = X_train / 255.0
X_test = X_test / 255.0

# Create the validation set
X_train, X_val, y_train, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

# One-Hot Encode the labels (required for categorical_crossentropy)
Y_train = to_categorical(y_train, num_classes=25)
Y_val = to_categorical(y_val, num_classes=25)
Y_test = to_categorical(y_test, num_classes=25)


print(f"Training set shape: {X_train.shape}")
print(f"Validation set shape: {X_val.shape}")
print(f"Test set shape: {X_test.shape}")
print(f"Number of classes: {len(np.unique(y_train))}")



Saving sign_mnist_test.csv to sign_mnist_test (1).csv
Saving sign_mnist_train.csv to sign_mnist_train (1).csv
Loading datasets...
Training set shape: (21964, 784)
Validation set shape: (5491, 784)
Test set shape: (7172, 784)
Number of classes: 24


# **Part 2:** Build the MLP Model Function

In [2]:
def build_mlp_model(n_hidden_layers, n_neurons, activation, learning_rate, optimizer_type, l2_reg = None):

  model = Sequential()

  # Define the kernel regularizer based on the l2_reg parameter
  regularizer = l2(l2_reg) if l2_reg else None

  # Loop to add the specified number of Hidden Layers
  for i in range(n_hidden_layers):
        model.add(Dense(n_neurons, activation = activation, kernel_regularizer=regularizer))

  # Add the final Output Layer
  model.add(Dense(25, activation = "softmax"))

  # Choose an optimizer
  if optimizer_type == "Adam":
      optimizer = Adam(learning_rate=learning_rate)
  elif optimizer_type == "SGD":
      optimizer = SGD(learning_rate=learning_rate)
  else:
      raise ValueError("Invalid optimizer type")


  # Compile the model
  model.compile(optimizer, loss = "categorical_crossentropy", metrics = ["accuracy"])

  return model

# **Part 3:** Train Using Varying Parameter Architecture. Evaluate Accuracy

In [23]:
# Experiment with 10 different parameter architecture
varying_param_tests = [
    # Test 1: Base Test
    {"hidden_layers": 2, "neurons": 128, "activation": "relu", "learning_rate": 0.001, "optimizer": "Adam", "l2_reg": None},
    # Test 2: Fewer Hidden Layers
    {"hidden_layers": 1, "neurons": 128, "activation": "relu", "learning_rate": 0.001, "optimizer": "Adam", "l2_reg": None},
    # Test 3: More Hidden Layers
    {"hidden_layers": 3, "neurons": 128, "activation": "relu", "learning_rate": 0.001, "optimizer": "Adam", "l2_reg": None},
    # Test 4: Fewer Neurons
    {"hidden_layers": 2, "neurons": 64, "activation": "relu", "learning_rate": 0.001, "optimizer": "Adam", "l2_reg": None},
    # Test 5: More Neurons
    {"hidden_layers": 2, "neurons": 256, "activation": "relu", "learning_rate": 0.001, "optimizer": "Adam", "l2_reg": None},
    # Test 6: Different Activation Function
    {"hidden_layers": 2, "neurons": 128, "activation": "sigmoid", "learning_rate": 0.001, "optimizer": "Adam", "l2_reg": None},
    # Test 7: Lower Learning Rate
    {"hidden_layers": 2, "neurons": 128, "activation": "relu", "learning_rate": 0.0001, "optimizer": "Adam", "l2_reg": None},
    # Test 8: Higher Learning Rate
    {"hidden_layers": 2, "neurons": 128, "activation": "relu", "learning_rate": 0.01, "optimizer": "Adam", "l2_reg": None},
    # Test 9: SGD Optimizer
    {"hidden_layers": 2, "neurons": 128, "activation": "relu", "learning_rate": 0.001, "optimizer": "SGD", "l2_reg": None},
    # Test 10: L2 Regularization
    {"hidden_layers": 2, "neurons": 128, "activation": "relu", "learning_rate": 0.001, "optimizer": "Adam", "l2_reg": 0.001}
]

results = []

print("--- RUNNING 10 TESTS. THIS MAY TAKE A WHILE (APPROX 6 MIN.) ---")

for i, params in enumerate(varying_param_tests):
  # Display current test being run
  print(f"\nRunning Test {i + 1}/10...")
  print(params)

  # Build model using current test's parameters
  model = build_mlp_model(
      n_hidden_layers = params["hidden_layers"],
      n_neurons = params["neurons"],
      activation = params["activation"],
      learning_rate = params["learning_rate"],
      optimizer_type = params["optimizer"],
      l2_reg = params["l2_reg"]
  )

  # Train the model
  history = model.fit(
      X_train,
      Y_train,
      validation_data=(X_val, Y_val),
      epochs=10,
      batch_size=32,
      verbose=0
    )

  # Evaluate the model on the test set
  test_loss, test_acc = model.evaluate(X_test, Y_test, verbose=0)

  # Save the results to a results List
  params["test_loss"] = test_loss
  params["test_accuracy"] = test_acc
  params["val_loss"] = history.history["val_loss"][-1]
  params["val_accuracy"] = history.history['val_accuracy'][-1]
  results.append(params)

  # Print results for each test
  print(f"Test Loss: {test_loss:.4f} | Test Accuracy: {test_acc:.4f} | Validation Loss: {history.history['val_loss'][-1]:.4f} | Validation Acc: {history.history['val_accuracy'][-1]:.4f}")

print("\n--- TESTS COMPLETE ---")

--- RUNNING 10 TESTS. THIS MAY TAKE A WHILE (APPROX 6 MIN.) ---

Running Test 1/10...
{'hidden_layers': 2, 'neurons': 128, 'activation': 'relu', 'learning_rate': 0.001, 'optimizer': 'Adam', 'l2_reg': None}
Test Loss: 1.1352 | Test Accuracy: 0.7476 | Validation Loss: 0.0708 | Validation Acc: 0.9769

Running Test 2/10...
{'hidden_layers': 1, 'neurons': 128, 'activation': 'relu', 'learning_rate': 0.001, 'optimizer': 'Adam', 'l2_reg': None}
Test Loss: 1.0510 | Test Accuracy: 0.6926 | Validation Loss: 0.3173 | Validation Acc: 0.9146

Running Test 3/10...
{'hidden_layers': 3, 'neurons': 128, 'activation': 'relu', 'learning_rate': 0.001, 'optimizer': 'Adam', 'l2_reg': None}
Test Loss: 1.3247 | Test Accuracy: 0.7281 | Validation Loss: 0.1987 | Validation Acc: 0.9193

Running Test 4/10...
{'hidden_layers': 2, 'neurons': 64, 'activation': 'relu', 'learning_rate': 0.001, 'optimizer': 'Adam', 'l2_reg': None}
Test Loss: 1.0304 | Test Accuracy: 0.7051 | Validation Loss: 0.2494 | Validation Acc: 0.92

# **Part 4:** Collect and Analyze the Results

In [25]:
results_df = pd.DataFrame(results)

results_df.insert(
    0,
    "test",
    [f"Test {i + 1}" for i in range(len(results_df))]
)

# Sort by test accuracy to find the most accurate test configuration
sorted_df = results_df.sort_values(by='val_accuracy', ascending=False).reset_index(drop=True)

print("\n--- Results Sorted by Validation Accuracy ---")
display(sorted_df)

highest_acc = sorted_df.iloc[0]
lowest_acc = sorted_df.iloc[-1]

print("\n--- Best Performing Configuration ---")
print(highest_acc)
print("\n--- Worst Performing Configuration ---")
print(lowest_acc)


--- Results Sorted by Validation Accuracy ---


,test,hidden_layers,neurons,activation,learning_rate,optimizer,l2_reg,test_loss,test_accuracy,val_loss,val_accuracy
0,Test 5,2,256,relu,0.0010,Adam,NaN,0.840717,0.820413,0.007955,1.000000
1,Test 6,2,128,sigmoid,0.0010,Adam,NaN,0.870712,0.749582,0.113179,0.983974
2,Test 1,2,128,relu,0.0010,Adam,NaN,1.135163,0.747630,0.070843,0.976871
3,Test 10,2,128,relu,0.0010,Adam,0.001,1.291478,0.721277,0.354906,0.960663
4,Test 4,2,64,relu,0.0010,Adam,NaN,1.030371,0.705103,0.249443,0.928064
5,Test 3,3,128,relu,0.0010,Adam,NaN,1.324681,0.728109,0.198687,0.919323
6,Test 2,1,128,relu,0.0010,Adam,NaN,1.050991,0.692554,0.317335,0.914587
7,Test 7,2,128,relu,0.0001,Adam,NaN,1.090079,0.665086,0.707590,0.813149
8,Test 9,2,128,relu,0.0010,SGD,NaN,2.223344,0.377858,2.175609,0.434347
9,Test 8,2,128,relu,0.0100,Adam,NaN,2.691967,0.156721,2.515625,0.153524



--- Best Performing Configuration ---
test               Test 5
hidden_layers           2
neurons               256
activation           relu
learning_rate       0.001
optimizer            Adam
l2_reg                NaN
test_loss        0.840717
test_accuracy    0.820413
val_loss         0.007955
val_accuracy          1.0
Name: 0, dtype: object

--- Worst Performing Configuration ---
test               Test 8
hidden_layers           2
neurons               128
activation           relu
learning_rate        0.01
optimizer            Adam
l2_reg                NaN
test_loss        2.691967
test_accuracy    0.156721
val_loss         2.515625
val_accuracy     0.153524
Name: 9, dtype: object
